# Phase 5 — UMAP Experiments

**Goal:** Understand how UMAP's two key hyperparameters control the embedding, compare it against every method from Phases 3–4, and benchmark its speed.

| Hyperparameter | Low value | High value |
|---|---|---|
| `n_neighbors` | Fine local detail, may miss global shape | Broad global structure, loses local clusters |
| `min_dist` | Tight clumps (0.0) | Evenly spread points (1.0) |

**Key rule:** always set `random_state=42` — UMAP is stochastic and results change every run without it.

**Kernel:** `manifold-discovery`

In [ ]:
# Cell 1 — Imports & path fix
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd())
while not (ROOT / 'environment.yml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT = {ROOT}  |  src found: {(ROOT / "src").exists()}')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from sklearn.manifold import trustworthiness

from src.datasets.generators import load_all_datasets
from src.methods.umap_method import (run_umap, umap_param_sweep,
                                      umap_metric_compare,
                                      umap_speed_benchmark,
                                      run_all_umap)
from src.utils.plotting import plot_2d, savefig
from src.config import CFG

FIGURES = CFG['figures_dir']
SEED    = CFG['random_seed']
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100})

In [ ]:
# Cell 2 — Load saved datasets
DATA_DIR = CFG['data_dir']
file_map = {
    'Swiss Roll':   'swiss_roll.npz',
    'S-Curve':      's_curve.npz',
    'Torus':        'torus.npz',
    'Mobius Strip': 'mobius_strip.npz',
}

datasets = {}
for name, fname in file_map.items():
    path = DATA_DIR / fname
    if path.exists():
        d = np.load(path)
        datasets[name] = (d['X'], d['t'])
        print(f'  Loaded {name}: X={d["X"].shape}')
    else:
        print(f'  {fname} not found — regenerating')
        datasets = load_all_datasets(n_samples=CFG['n_samples'], noise=0.1, seed=SEED)
        break

In [ ]:
# Cell 3 — UMAP default run on all four datasets
# n_neighbors=15, min_dist=0.1 — the standard starting point

print('Running UMAP (default params) on all datasets...')
results_default = run_all_umap(datasets, n_neighbors=15, min_dist=0.1,
                                random_state=SEED)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('UMAP — default params (n_neighbors=15, min_dist=0.1)', fontsize=13)

for ax, (name, (X, t)) in zip(axes.flat, datasets.items()):
    X_emb, meta = results_default[name]
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1], c=t, cmap='viridis', s=5, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='t')
    ax.set_title(f'{name}  ({meta["fit_time_s"]:.2f}s)', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '04_umap_default_all.png')
plt.show()

In [ ]:
# Cell 4 — Hyperparameter sweep: n_neighbors x min_dist on Swiss Roll
# This is the most important cell in Phase 5 — shows the core UMAP trade-off

X_sr, t_sr = datasets['Swiss Roll']

neighbor_values = [5, 15, 30, 50]
dist_values     = [0.0, 0.1, 0.5, 1.0]

print('Running 4x4 sweep (16 UMAP fits)...')
sweep = umap_param_sweep(X_sr, t_sr,
                          neighbor_values=neighbor_values,
                          dist_values=dist_values,
                          random_state=SEED)

fig, axes = plt.subplots(len(neighbor_values), len(dist_values),
                          figsize=(16, 16))
fig.suptitle('UMAP hyperparameter sweep — Swiss Roll\n'
             'Rows: n_neighbors  |  Columns: min_dist', fontsize=13, y=1.01)

for r, k in enumerate(neighbor_values):
    for c, d in enumerate(dist_values):
        ax    = axes[r, c]
        res   = sweep[(k, d)]
        X_emb = res['X_emb']
        sc    = ax.scatter(X_emb[:, 0], X_emb[:, 1],
                           c=t_sr, cmap='viridis', s=4, alpha=0.8)
        title_color = '#085041' if not res['meta']['error_msg'] else '#A32D2D'
        ax.set_title(f'k={k}  d={d}', fontsize=9, color=title_color)
        ax.set_xticks([]); ax.set_yticks([])
        # Column headers
        if r == 0:
            ax.set_xlabel(f'min_dist={d}', fontsize=10, labelpad=6)
        # Row labels
        if c == 0:
            ax.set_ylabel(f'n_neighbors={k}', fontsize=10)

plt.tight_layout()
savefig(fig, FIGURES / '04_umap_sweep_swiss_roll.png')
plt.show()
print()
print('Key insight:')
print('  min_dist=0.0 → tight clusters    min_dist=1.0 → evenly spread')
print('  n_neighbors small → local detail  n_neighbors large → global shape')

In [ ]:
# Cell 5 — min_dist effect in isolation (n_neighbors=15 fixed)
# Clean 1x4 strip showing just the min_dist axis

dist_values = [0.0, 0.1, 0.5, 1.0]
fig, axes   = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('UMAP: effect of min_dist (n_neighbors=15 fixed) — Swiss Roll', fontsize=12)

for ax, d in zip(axes, dist_values):
    X_emb, meta = run_umap(X_sr, n_neighbors=15, min_dist=d, random_state=SEED)
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1], c=t_sr, cmap='viridis', s=5, alpha=0.8)
    ax.set_title(f'min_dist={d}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '04_umap_min_dist_effect.png')
plt.show()
print('min_dist=0.0: points cluster tightly — good for finding groups')
print('min_dist=1.0: points spread evenly  — good for global topology')

In [ ]:
# Cell 6 — Distance metric comparison on Swiss Roll

print('Comparing metrics...')
metric_results = umap_metric_compare(X_sr, n_neighbors=15, min_dist=0.1,
                                      random_state=SEED)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('UMAP: distance metric comparison — Swiss Roll (n_neighbors=15, min_dist=0.1)', fontsize=11)

for ax, (metric, (X_emb, meta)) in zip(axes, metric_results.items()):
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1], c=t_sr, cmap='viridis', s=5, alpha=0.8)
    ax.set_title(f'metric={metric}\n{meta["fit_time_s"]:.2f}s', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '04_umap_metrics.png')
plt.show()
print('Euclidean is best for these Euclidean point clouds.')
print('Cosine metric becomes relevant for text embeddings or high-dim sparse data.')

In [ ]:
# Cell 7 — Speed benchmark: UMAP vs Isomap vs LLE

print('Benchmarking fit times...')
bench_rows = umap_speed_benchmark(datasets, random_state=SEED)

df_bench = pd.DataFrame(bench_rows).set_index('Dataset')
print('\nFit times (seconds):')
print(df_bench.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x       = np.arange(len(df_bench))
w       = 0.25
cols    = ['UMAP (s)', 'Isomap (s)', 'LLE (s)']
colors  = ['#378ADD', '#1D9E75', '#BA7517']

for i, (col, color) in enumerate(zip(cols, colors)):
    ax.bar(x + i * w, df_bench[col], w, label=col.replace(' (s)', ''), color=color, alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels(df_bench.index, fontsize=10)
ax.set_ylabel('Fit time (seconds)', fontsize=11)
ax.set_title('Speed: UMAP vs Isomap vs LLE', fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig(fig, FIGURES / '04_umap_speed_benchmark.png')
plt.show()

In [ ]:
# Cell 8 — UMAP on noisy data: robustness test
# Re-generate Swiss Roll at three noise levels and compare

from src.datasets.generators import swiss_roll

noise_levels = CFG['noise_levels']   # [0.0, 0.1, 0.3]
fig, axes    = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('UMAP noise robustness — Swiss Roll (n_neighbors=15, min_dist=0.1)', fontsize=12)

for ax, noise in zip(axes, noise_levels):
    X_noisy, t_noisy = swiss_roll(n_samples=CFG['n_samples'], noise=noise, seed=SEED)
    X_emb, meta      = run_umap(X_noisy, n_neighbors=15, min_dist=0.1, random_state=SEED)
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1], c=t_noisy, cmap='viridis', s=5, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='t')
    ax.set_title(f'noise={noise}  ({meta["fit_time_s"]:.2f}s)', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '04_umap_noise_robustness.png')
plt.show()
print('UMAP handles noise well — structure is preserved even at sigma=0.3.')

In [ ]:
# Cell 9 — Trustworthiness scores + final summary table

rows = []
for name, (X, t) in datasets.items():
    X_emb, meta = results_default[name]
    valid = np.std(X_emb) > 1e-6
    tw    = trustworthiness(X, X_emb, n_neighbors=10) if valid else None
    rows.append({
        'Dataset':       name,
        'n_neighbors':   meta['n_neighbors'],
        'min_dist':      meta['min_dist'],
        'Trustworthiness': f'{tw:.3f}' if tw else 'FAIL',
        'Fit time (s)':  f'{meta["fit_time_s"]:.3f}' if meta['fit_time_s'] else 'n/a',
    })

df = pd.DataFrame(rows).set_index('Dataset')
print('UMAP results summary:')
print(df.to_string())

csv_path = ROOT / 'results' / 'metrics' / '04_umap_metrics.csv'
df.to_csv(csv_path)
df_bench.to_csv(ROOT / 'results' / 'metrics' / '04_umap_speed_benchmark.csv')
print(f'\nSaved -> results/metrics/04_umap_metrics.csv')
print(f'Saved -> results/metrics/04_umap_speed_benchmark.csv')

## Phase 5 Summary

| Finding | Detail |
|---|---|
| UMAP on Swiss Roll | Clean unrolling — comparable to Isomap, faster |
| UMAP on S-Curve | Clean arc — consistent with LLE Modified |
| UMAP on Torus | Ring structure — collapses minor angle like all methods |
| UMAP on Möbius | Circle — non-orientable structure partially preserved |
| Best params (default) | n_neighbors=15, min_dist=0.1 |
| min_dist effect | 0.0 = tight clusters, 1.0 = uniform spread |
| n_neighbors effect | Small = local detail, large = global topology |
| Speed vs others | UMAP ≈ LLE speed, significantly faster than Isomap on large n |
| Noise robustness | Strong — handles sigma=0.3 without major degradation |
| Best metric | Euclidean for geometric point clouds |

**Next:** `05_autoencoder.ipynb` — Phase 6, deep learning approach with PyTorch.